In [1]:
# Setup dataset
import numpy as np
from dynamics_testing import sim, attractor_dist, make_train

SEG, SETTLE, REENTRAIN, N_AUTO, DISCARD, WARM_TRAIN = 2000, 1000, 50, 1200, 200, 200
STATES = ["chaotic", "periodic_A", "periodic_B"]
RS_ROT = 700
N_STEPS = SETTLE + RS_ROT + N_AUTO

def rhs(_, y, b):
    x, yy, z = y
    return [np.sin(yy) - b * x, np.sin(z) - b * yy, np.sin(x) - b * z]

TRAIN = multi_t[:, :4 * SEG] if "multi_t" in globals() else make_train(rhs, SEG)

def make_ref(n_ics, n_steps, seed):
    rng = np.random.default_rng(seed)
    out = {"chaotic": [sim(rhs, 0.18, n_steps, rng.uniform(-4, 4, 3)) for _ in range(n_ics)], "periodic_A": [], "periodic_B": []}
    ex = None
    while min(len(out["periodic_A"]), len(out["periodic_B"])) < n_ics:
        tr = sim(rhs, 0.29, n_steps, rng.uniform(-4, 4, 3)); ex = tr if ex is None else ex
        k = "periodic_A" if attractor_dist(tr, ex) < 0.1 else "periodic_B"
        if len(out[k]) < n_ics: out[k].append(tr)
    return {k: np.concatenate(v, axis=1) for k, v in out.items()}

REF = make_ref(2, 5000, seed=300)
ref_label = lambda tr: min(("periodic_A", "periodic_B"), key=lambda k: attractor_dist(tr, REF[k]))

def make_states(n_ics, n_steps, seed):
    rng = np.random.default_rng(seed)
    out = {"chaotic": [sim(rhs, 0.18, n_steps, rng.uniform(-4, 4, 3)) for _ in range(n_ics)], "periodic_A": [], "periodic_B": []}
    while min(len(out["periodic_A"]), len(out["periodic_B"])) < n_ics:
        tr = sim(rhs, 0.29, n_steps, rng.uniform(-4, 4, 3)); k = ref_label(tr)
        if len(out[k]) < n_ics: out[k].append(tr)
    return out

VAL, TEST = make_states(4, N_STEPS, seed=100), make_states(4, N_STEPS, seed=200)

In [2]:
# Setup model testing harnesses
from testing_harness import get_harness

RS_OMEGA0 = 0.01
RN_BASE = dict(input_dims=3, spectral_radius=0.6, input_weight=1.2, average_degree_nodes=10, num_nodes=100,
               leakage=0, omega0=RS_OMEGA0, epsilon1=-0.2, epsilon2=0.6,
               mean_phase_tolerance=RS_OMEGA0,          # >= one rotation step, so the pi-crossing can't be skipped
               error_tolerance=np.inf)                  # freeze on phase alone
SEEDS = list(range(8))
HARNESS = get_harness()

In [3]:
# Spiking, RS, ESN recall comparison + fidelity (lambda_max, attractor distances) in one pass
import os, time, copy
import pandas as pd
from scipy.stats import binomtest
from dynamics_testing import valid_time, classify, attractor_dist
from testing_harness import ALPHAS

OUT = "recall_final.csv"                         # Chua: "recall_final-chua.csv"   (fresh file: new columns)
TRUE_LAMBDA = 0.035                              # Chua: 0.021 (per frame)
N_LYAP, EPS = 600, 1e-8
RS_TURN = int(np.ceil(2 * np.pi / RS_OMEGA0))

MODELS = {"spike_c1.0": ("spk", 1.0),
          "spike_c0":   ("spk", 0.0),            # ESN control sharing W, W_in with spike_c1.0
          "RN_pub":     ("rs",  0.6),
          "RN_c0":      ("rs",  0.0)}            # ESN control sharing W, W_in with RN_pub

def closed_loop_twin(warm, W_out, seed=0):
    """Closed-loop prediction from a frozen, re-entrained state, plus a twin started EPS away -> lambda_max."""
    w = copy.deepcopy(warm); w.output_weights = W_out
    v = copy.deepcopy(w)
    v.node_states = v.node_states + EPS * np.random.default_rng(seed).standard_normal(v.node_states.shape)
    P, logd = [], []
    for k in range(max(N_AUTO, N_LYAP)):
        out = W_out @ w.node_states
        P.append(out)
        w.advance_nodes(out, save_history=False)
        if k < N_LYAP:
            v.advance_nodes(W_out @ v.node_states, save_history=False)
            logd.append(np.log(np.linalg.norm(w.node_states - v.node_states) + 1e-300))
    logd = np.asarray(logd)
    grow = np.flatnonzero(logd < np.log(1e-3))[:200]
    lam = np.polyfit(grow, logd[grow], 1)[0] if len(grow) > 20 else np.nan
    return np.asarray(P[:N_AUTO]).T, lam

def warm_spk(warm, x):                           # settled + frozen -> re-entrained (what run_frozen does first)
    w = copy.deepcopy(warm)
    for t in range(SETTLE, SETTLE + REENTRAIN):
        w.advance_nodes(x[:, t], save_history=False)
    return w, SETTLE + REENTRAIN

def warm_rs(net, x, W_out):                      # settle, A6 search, freeze, re-entrain (what predict_search does first)
    w = copy.deepcopy(net); w.output_weights = W_out
    w.predict_search(x, SETTLE, REENTRAIN, prediction_time=0)
    return w, SETTLE + RS_TURN + REENTRAIN

done = set(map(tuple, pd.read_csv(OUT)[["model", "seed"]].drop_duplicates().values)) if os.path.exists(OUT) else set()
for name, (kind, c) in MODELS.items():
    train_fn, settle_fn = HARNESS[kind]
    for seed in SEEDS:
        if (name, seed) in done:
            continue
        t0, rows = time.time(), []
        net, W = train_fn(RN_BASE, seed, c, TRAIN, WARM_TRAIN)
        for split, sets in (("val", VAL), ("test", TEST)):
            for state, trajs in sets.items():
                for i, x in enumerate(trajs):
                    settled, _, _, froze, cost = settle_fn(net, x, SETTLE, REENTRAIN, offset=1 + i)
                    if kind == "spk":
                        warm, ts = warm_spk(settled, x)
                    truth = x[:, ts:ts + N_AUTO] if kind == "spk" else None
                    for a in ALPHAS:
                        if kind == "rs":
                            warm, ts = warm_rs(net, x, W[a])                # RS freeze depends on W_out (A6)
                        p, lam = closed_loop_twin(warm, W[a], seed=i)
                        truth = x[:, ts:ts + N_AUTO]
                        label, d = classify(p[:, DISCARD:], REF)
                        rows.append(dict(model=name, seed=seed, split=split, alpha=a, state=state, ic=i,
                                         pred=label, correct=label == state, vpt=valid_time(p, truth),
                                         lam=lam, d_true=attractor_dist(truth[:, DISCARD:], REF[state]),
                                         froze=froze, **{f"d_{k}": v for k, v in d.items()},
                                         **{k: cost[k] for k in
                                            ("event_msgs", "dense_msgs", "msg_ratio", "rate_mean", "frac_silent")}))
        pd.DataFrame(rows).to_csv(OUT, mode="a", header=not os.path.exists(OUT), index=False)
        print(f"{name} seed {seed}: {time.time() - t0:.0f}s", flush=True)

# ======================= analysis =======================
df = pd.read_csv(OUT); df["correct"] = df.correct.astype(bool)
df = df.drop_duplicates(["model", "seed", "split", "alpha", "state", "ic"], keep="last")

def scores(g):
    acc = g.groupby("state").correct.mean()
    per = g[g.state != "chaotic"]
    return pd.Series(dict(
        S=0.5 * acc["chaotic"] + 0.25 * acc["periodic_A"] + 0.25 * acc["periodic_B"],
        regime=0.5 * (g[g.state == "chaotic"].pred == "chaotic").mean()
               + 0.5 * per.pred.isin(["periodic_A", "periodic_B"]).mean(),
        orbit=(per.pred == per.state).mean()))

BEST = (df[df.split == "val"].groupby(["model", "alpha"]).apply(scores, include_groups=False).S
        .groupby("model").idxmax().map(lambda k: k[1]).to_dict())
test = df[(df.split == "test") & (df.alpha == df.model.map(BEST))]
print("alpha (val):", BEST)

acc = test.groupby(["model", "state"]).correct.agg(["sum", "count"])
print((acc["sum"].astype(int).astype(str) + "/" + acc["count"].astype(str)).unstack())

# S with seed-bootstrap CIs (seeds resampled together; paired differences use the same draws)
rng = np.random.default_rng(0)
seeds = np.sort(test.seed.unique()); by_seed = {s: g for s, g in test.groupby("seed")}
S_point = test.groupby("model").apply(lambda m: scores(m).S, include_groups=False)
B = pd.DataFrame([pd.concat([by_seed[s] for s in rng.choice(seeds, len(seeds))])
                  .groupby("model").apply(lambda m: scores(m).S, include_groups=False) for _ in range(10_000)])
print(pd.DataFrame({"S": S_point, "lo": B.quantile(0.025), "hi": B.quantile(0.975)}).round(3))
for a, b in (("spike_c1.0", "spike_c0"), ("spike_c1.0", "RN_pub"), ("RN_pub", "RN_c0")):
    dS = B[a] - B[b]
    print(f"S({a}) - S({b}) = {S_point[a] - S_point[b]:.3f} [{dS.quantile(0.025):.3f}, {dS.quantile(0.975):.3f}]")

# fidelity on the chaotic tests
ch = test[test.state == "chaotic"].assign(d_ratio=lambda g: g.d_chaotic / g.d_true)
print(f"true lambda_max = {TRUE_LAMBDA}")
print(ch.groupby("model").agg(lam_mean=("lam", "mean"), lam_median=("lam", "median"),
                              frac_chaotic=("lam", lambda v: np.mean(v > TRUE_LAMBDA / 2)),
                              d_ratio_median=("d_ratio", lambda v: v[np.isfinite(v)].median())).round(3))

# paired McNemar + seed bootstrap on accuracy (kept for the text)
def paired(a, b, n_boot=10_000, rng=np.random.default_rng(0)):
    key = ["seed", "state", "ic"]
    A = test[test.model == a].set_index(key).correct
    Bc = test[test.model == b].set_index(key).correct
    A, Bc = A.align(Bc, join="inner")
    a_only, b_only = int((A & ~Bc).sum()), int((~A & Bc).sum())
    p = binomtest(a_only, a_only + b_only, 0.5).pvalue if a_only + b_only else 1.0
    return dict(A=a, B=b, n=len(A), A_only=a_only, B_only=b_only, mcnemar_p=p)
print(pd.DataFrame([paired("spike_c1.0", "RN_pub"), paired("spike_c1.0", "spike_c0"),
                    paired("RN_pub", "RN_c0")]).round(3).to_string(index=False))

cost = test.drop_duplicates(["model", "seed", "state", "ic"])
print(cost.groupby("model")[["rate_mean", "event_msgs", "dense_msgs", "msg_ratio"]].mean().round(4))

spike_c1.0 seed 5: 1312s
spike_c1.0 seed 6: 1299s
spike_c1.0 seed 7: 1431s
spike_c0 seed 0: 1192s
spike_c0 seed 1: 1137s
spike_c0 seed 2: 1136s
spike_c0 seed 3: 1130s
spike_c0 seed 4: 1131s
spike_c0 seed 5: 1135s
spike_c0 seed 6: 1132s
spike_c0 seed 7: 1133s
RN_pub seed 0: 162s
RN_pub seed 1: 163s
RN_pub seed 2: 162s
RN_pub seed 3: 162s
RN_pub seed 4: 161s
RN_pub seed 5: 162s
RN_pub seed 6: 162s
RN_pub seed 7: 162s
RN_c0 seed 0: 162s
RN_c0 seed 1: 162s
RN_c0 seed 2: 163s
RN_c0 seed 3: 161s
RN_c0 seed 4: 161s
RN_c0 seed 5: 165s
RN_c0 seed 6: 160s
RN_c0 seed 7: 161s
alpha (val): {'RN_c0': 0.1, 'RN_pub': 0.1, 'spike_c0': 0.1, 'spike_c1.0': 0.01}
state      chaotic periodic_A periodic_B
model                                   
RN_c0        24/32      20/32      20/32
RN_pub        2/32      27/32      30/32
spike_c0     23/32      20/32      20/32
spike_c1.0   29/32      29/32      31/32
                S     lo     hi
model                          
RN_c0       0.688  0.547  0.844
RN_pub 

In [4]:
# Unsupervised detection: no labels, each detector calibrated on its own pre-switch window.
# Spiking and RS link layers need no W_out (training doesn't change link dynamics); the ESN detector does.
from tqdm import tqdm
import scipy.sparse as sparse
from testing_harness import RN
from spike_sharing import SpikingNetwork
from scipy.integrate import solve_ivp

N_PRE, N_POST = 2500, 2000
T_REF0, T_REF1, T_SW = 1000, 2000, N_PRE             # settle | reference | false-alarm window | switch
DET_ICS, TAU_S, PERSIST, KS = 4, 20, 10, (3, 5)
DIRS = {"chaos->periodic": (0.18, 0.29), "periodic->chaos": (0.29, 0.18)}

def make_switch(b0, b1, init):
    y = sim(rhs, b0, N_PRE, init)
    sol = solve_ivp(rhs, (0, N_POST), y[:, -1], args=(b1,), t_eval=np.arange(1, N_POST + 1), rtol=1e-9, atol=1e-9)
    return np.concatenate([y, sol.y], axis=1)

rng = np.random.default_rng(700)
STREAMS = {d: [make_switch(b0, b1, rng.uniform(-4, 4, 3)) for _ in range(DET_ICS)] for d, (b0, b1) in DIRS.items()}

def ema(s, tau):
    out, m = np.empty_like(s), s[0]
    for i, v in enumerate(s):
        m += (v - m) / tau; out[i] = m
    return out

def detect(s, k):
    s = ema(np.asarray(s, float), TAU_S)
    mu, sd = s[T_REF0:T_REF1].mean(), s[T_REF0:T_REF1].std() + 1e-12
    run = np.convolve((np.abs(s - mu) > k * sd).astype(int), np.ones(PERSIST, int))[:len(s)] >= PERSIST
    post = np.flatnonzero(run[T_SW:])
    return bool(run[T_REF1:T_SW].any()), (post[0] if len(post) else np.nan)
TAU_CTX = 60                                          # = spiking mod_tau, so all context signals share one filter

def ref_dist(X):                                      # RMS distance of each row from its reference-window mean
    return np.sqrt(np.mean(np.abs(X - X[T_REF0:T_REF1].mean(axis=0)) ** 2, axis=1))

def rel_phasor(phases):                               # e^{i(theta_i - psi)}, psi = global mean phase
    return np.exp(1j * (phases - np.angle(np.mean(np.exp(1j * phases)))))

det_rows, SIG = [], {}
for seed in SEEDS:
    t0 = time.time()
    spk = SpikingNetwork(model_seed=seed)                                  # final model, c = 1.0
    rs = RN(**RN_BASE, link_strength_change_ratio=0.6, model_seed=seed); rs.input_weights = spk.input_weights.copy()
    esn = RN(**RN_BASE, link_strength_change_ratio=0.0, model_seed=seed); esn.input_weights = spk.input_weights.copy()
    esn.train(TRAIN, warmup_time=WARM_TRAIN)                               # == spike_c0 (verified)
    esn.regularization, esn.lsqrs = BEST["spike_c0"], False; esn.compute_weights(); esn.reset_history()

    for d, xs in tqdm(STREAMS.items()):
        for i, x in enumerate(xs):
            T = x.shape[1]
            spk.reset_initial_states(1 + i); spk.reset_history()
            rs.reset_initial_states(seed_offset=1 + i); rs.reset_history()
            esn.reset_initial_states(seed_offset=1 + i); esn.reset_history()
            R_spk, R_rs, err = np.empty(T), np.empty(T), np.full(T, np.nan)
            ctx = np.empty((T, spk.num_links), np.complex64)
            ctx_rs = np.empty((T, rs.num_links), np.complex64)
            st_esn = np.empty((T, esn.num_nodes))
            z_rs, n_lp = None, None
            for t in range(T):
                spk.advance(x[:, t], save_history=False)
                rs.advance(x[:, t], save_history=False)
                esn.advance(x[:, t], save_history=False)
                r = rel_phasor(rs.link_states)
                z_rs = r if z_rs is None else z_rs + (r - z_rs) / TAU_CTX
                ctx_rs[t] = z_rs
                n_lp = esn.node_states.copy() if n_lp is None else n_lp + (esn.node_states - n_lp) / TAU_CTX
                st_esn[t] = n_lp
                R_spk[t] = np.abs(np.mean(np.exp(1j * spk.link_phase)))
                ctx[t] = spk.rel_lp
                R_rs[t] = np.abs(np.mean(np.exp(1j * rs.link_states)))
                if t + 1 < T:
                    err[t] = np.linalg.norm(esn.output_weights @ esn.node_states - x[:, t + 1])
            err[-1] = err[-2]
            ref = ctx[T_REF0:T_REF1].mean(axis=0)
            sig = {"spike_R": R_spk, "spike_ctx": ref_dist(ctx),
                   "RS_R": R_rs, "RS_ctx": ref_dist(ctx_rs),
                   "ESN_err": err, "ESN_state": ref_dist(st_esn)}
            SIG[(seed, d, i)] = sig
            for name, s in sig.items():
                for k in KS:
                    fa, lat = detect(s, k)
                    det_rows.append(dict(seed=seed, dir=d, ic=i, detector=name, k=k, false_alarm=fa, latency=lat))
    print(f"seed {seed}: {time.time() - t0:.0f}s", flush=True)

det = pd.DataFrame(det_rows); det.to_csv("detection.csv", index=False); pd.to_pickle(SIG, "detection_signals.pkl")
det["hit"] = ~det.false_alarm & det.latency.notna()
summ = det.groupby(["dir", "detector", "k"]).agg(hit=("hit", "mean"), false_alarm=("false_alarm", "mean"))
summ["lat_median"] = det[det.hit].groupby(["dir", "detector", "k"]).latency.median()
summ["lat_iqr"] = det[det.hit].groupby(["dir", "detector", "k"]).latency.agg(lambda v: v.quantile(.75) - v.quantile(.25))
print(summ.round(2).to_string())

100%|███████████████████████████████████████████████████████████████████████████████████| 2/2 [20:19<00:00, 609.88s/it]

seed 0: 1228s



100%|███████████████████████████████████████████████████████████████████████████████████| 2/2 [20:04<00:00, 602.45s/it]

seed 1: 1213s



100%|███████████████████████████████████████████████████████████████████████████████████| 2/2 [20:08<00:00, 604.40s/it]

...

seed 5: 1211s



100%|███████████████████████████████████████████████████████████████████████████████████| 2/2 [20:04<00:00, 602.23s/it]

seed 6: 1213s



100%|███████████████████████████████████████████████████████████████████████████████████| 2/2 [20:07<00:00, 603.61s/it]

seed 7: 1215s
                              hit  false_alarm  lat_median  lat_iqr
dir             detector  k                                        
chaos->periodic ESN_err   3  0.91         0.09        29.0    39.00
                          5  0.91         0.00        67.0    84.00
                ESN_state 3  0.50         0.50        87.0     3.00
                          5  0.75         0.25       121.0    63.50
                RS_R      3  0.53         0.47       111.0    58.00
                          5  0.75         0.25       106.5    75.25
                RS_ctx    3  0.56         0.44       120.5    72.00
                          5  0.75         0.25       119.0    89.25
                spike_R   3  0.34         0.00       297.0   198.00
                          5  0.00         0.00         NaN      NaN
                spike_ctx 3  0.38         0.62        93.5     3.25
                          5  0.81         0.19       118.5    50.50
periodic->chaos ESN_err   3  1.00 

In [5]:
SIG = pd.read_pickle("detection_signals.pkl")
KGRID = np.round(np.arange(1, 12.01, 0.25), 2)
g = pd.DataFrame([dict(seed=s, dir=d, ic=i, detector=n, k=k, **dict(zip(("fa", "lat"), detect(x, k))))
                  for (s, d, i), sig in SIG.items() for n, x in sig.items() for k in KGRID])
g["hit"] = ~g.fa & g.lat.notna()
far = g.groupby(["detector", "k"]).fa.mean()                        # pooled over both directions: detector doesn't know which
K = {n: (f[f <= 0.05].index.get_level_values("k").min()) for n, f in far.groupby(level="detector")}
m = g[g.k == g.detector.map(K)]
res = m.groupby(["dir", "detector"]).agg(fa=("fa", "mean"), hit=("hit", "mean"))
res["lat_med"] = m[m.hit].groupby(["dir", "detector"]).lat.median()
print(K); print(res.round(2))

{'ESN_err': np.float64(3.0), 'ESN_state': np.float64(5.75), 'RS_R': nan, 'RS_ctx': nan, 'spike_R': np.float64(2.75), 'spike_ctx': np.float64(5.5)}
                             fa   hit  lat_med
dir             detector                      
chaos->periodic ESN_err    0.09  0.91     29.0
                ESN_state  0.03  0.97    139.0
                spike_R    0.06  0.31    299.5
                spike_ctx  0.09  0.91    125.0
periodic->chaos ESN_err    0.00  1.00      8.0
                ESN_state  0.00  1.00     13.0
                spike_R    0.03  0.97     27.0
                spike_ctx  0.00  1.00     42.0


In [6]:
from testing_harness import spk_settle, run_frozen
import copy

A_SW, N_STARTS, NICS = BEST["spike_c1.0"], 10, len(TEST["chaotic"])
reg = lambda s: "periodic" if str(s).startswith("periodic") else s
sw, bs = [], []

for seed in SEEDS:
    t0 = time.time()
    net = SpikingNetwork(model_seed=seed, link_strength_change_ratio=1.0, regularization=A_SW)
    net.train(TRAIN, warmup_time=WARM_TRAIN)
    W_out, H = net.output_weights.copy(), np.asarray(net.node_states_history).copy()   # H[t] <-> TRAIN[WARM_TRAIN + t]
    net.reset_history()
    seg = np.where(((WARM_TRAIN + np.arange(len(H))) // SEG) % 2 == 1, "train_periodic", "train_chaotic")

    warm, CTX = {}, {}
    for s in tqdm(STATES):
        for i, x in enumerate(TEST[s]):
            warm[(s, i)] = spk_settle(net, x, SETTLE, REENTRAIN, offset=1 + i)[0]
            CTX[(s, i)] = warm[(s, i)].rel_lp.copy()
    z0 = np.zeros_like(CTX[("chaotic", 0)])                       # neutral: m = 1 - c/2 on every link
    rng = np.random.default_rng(1000 + seed)

    # swap: drive and settle from one state, context implanted from another (different IC when same state)
    for d in tqdm(STATES):
        for i, x in enumerate(TEST[d]):
            conds = [("state", c, CTX[(c, (i + 1) % NICS if c == d else i)]) for c in STATES]
            conds += [("own", d, CTX[(d, i)]), ("shuffled", "none", rng.permutation(CTX[(d, i)])), ("neutral", "none", z0)]
            for kind, c, z in conds:
                w = copy.deepcopy(warm[(d, i)]); w.rel_lp = z.copy()
                p, _ = run_frozen(w, x, SETTLE, W_out, REENTRAIN, N_AUTO)
                lab = classify(p[:, DISCARD:], REF)[0]
                sw.append(dict(seed=seed, drive=d, ctx_kind=kind, ctx_state=c, pred=lab,
                               follows_ctx=lab == c, follows_drive=lab == d,
                               follows_ctx_regime=reg(lab) == reg(c)))

    # basin: closed loop from a node state under a fixed context, no drive at all
    for c, z in tqdm([(s, CTX[(s, i)]) for s in STATES for i in range(NICS)] + [("neutral", z0)]):
        for _ in range(N_STARTS):
            t = rng.integers(len(H))
            for start, n0 in ((seg[t], H[t]), ("gaussian", H.std() * rng.standard_normal(H.shape[1]))):
                w = copy.deepcopy(net); w.rel_lp, w.frozen, w.node_states = z.copy(), True, n0.copy()
                p, _ = run_frozen(w, None, 0, W_out, 0, N_AUTO)
                bs.append(dict(seed=seed, ctx=c, start=start, pred=classify(p[:, DISCARD:], REF)[0]))
    print(f"seed {seed}: {time.time() - t0:.0f}s", flush=True)

sw, bs = pd.DataFrame(sw), pd.DataFrame(bs)
sw.to_csv("swap.csv", index=False); bs.to_csv("basin.csv", index=False)

st = sw[sw.ctx_kind == "state"]; off = st[st.ctx_state != st.drive]
print(pd.crosstab([st.drive, st.ctx_state], st.pred))
print(off.groupby(["drive", "ctx_state"])[["follows_ctx", "follows_ctx_regime", "follows_drive"]].mean().round(2))
print(sw[sw.ctx_kind.isin(["own", "shuffled", "neutral"])].groupby(["ctx_kind", "drive"]).follows_drive.mean().unstack().round(2))

bs["ctx_group"] = bs.ctx.map(reg)
print(pd.crosstab([bs.start, bs.ctx], bs.pred, normalize="index").round(2))
ps = bs.assign(chaos=bs.pred == "chaotic").groupby(["seed", "ctx_group"]).chaos.mean().unstack()
sel = ps["chaotic"] - ps["periodic"]
boot = [sel.sample(len(sel), replace=True, random_state=k).mean() for k in range(10_000)]
print(ps.mean().round(2).to_dict(),
      f"| select = {sel.mean():.2f} [{np.percentile(boot, 2.5):.2f}, {np.percentile(boot, 97.5):.2f}]")

100%|██████████████████████████████████████████████████████████████████████████████████| 13/13 [01:03<00:00,  4.91s/it]

seed 0: 733s



100%|██████████████████████████████████████████████████████████████████████████████████| 13/13 [01:05<00:00,  5.00s/it]

seed 1: 815s



100%|██████████████████████████████████████████████████████████████████████████████████| 13/13 [01:21<00:00,  6.28s/it]

...


100%|██████████████████████████████████████████████████████████████████████████████████| 13/13 [01:06<00:00,  5.09s/it]

seed 6: 754s



100%|██████████████████████████████████████████████████████████████████████████████████| 13/13 [01:01<00:00,  4.75s/it]

seed 7: 751s


pred                   chaotic  none  periodic_A  periodic_B
drive      ctx_state                                        
chaotic    chaotic          29     0           0           3
           periodic_A        6     1           8          17
           periodic_B        1     0          11          20
periodic_A chaotic          27     0           5           0
           periodic_A        3     0          29           0
           periodic_B        1     0          31           0
periodic_B chaotic          27     0           0           5
           periodic_A        3     0           0          29
           periodic_B        1     0           0          31
                       follows_ctx  follows_ctx_regime  follows_drive
drive      ctx_state                                                 
chaotic    periodic_A         0.25                0.78           0.19
           periodic_B         0.62                0.97           0.03
periodic_A chaotic            0.84               